necessary libraries

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.interpolate import interp1d

loading baseline files

In [14]:
BASE = Path("/Users/muskaanchugh/Desktop/art_synthetic_data_gen")
BASELINE = BASE / "baseline_data"
INTERPOLATED = BASE / "interpolated_data"
INTERPOLATED.mkdir(exist_ok=True)

In [15]:
baseline_files = {f.stem: f for f in BASELINE.glob("*.csv")}

defining the interpolation function

In [16]:
MONTH_MAP = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12
}
MONTH_INV = {v: k for k, v in MONTH_MAP.items()}

def interpolate_drug(df, start="2015-08", end="2024-03"):
    df = df.copy()
    df["month_num"] = df["month"].str.lower().map(MONTH_MAP)
    df["t"] = (df["year"] - 2015) * 12 + (df["month_num"] - 1)

    linear = interp1d(df["t"].values, df["patients"].values)

    t_start = (int(start[:4]) - 2015) * 12 + (int(start[5:]) - 1)
    t_end = (int(end[:4]) - 2015) * 12 + (int(end[5:]) - 1)
    t_range = np.arange(t_start, t_end + 1)

    patients_interp = np.maximum(0, np.round(linear(t_range)).astype(int))

    return pd.DataFrame({
        "month": [MONTH_INV[m] for m in (t_range % 12 + 1)],
        "year": 2015 + t_range // 12,
        "patients": patients_interp
    })

generating interpolations

In [17]:
interpolated_dfs = {}

for name, path in baseline_files.items():
    df = pd.read_csv(path)
    interpolated_dfs[name] = interpolate_drug(df)
    new_name = name.replace("synthetic_baseline", "synthetic_interpolated") + ".csv"
    interpolated_dfs[name].to_csv(INTERPOLATED / new_name, index=False)